In [ ]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
import os
from sentence_transformers import SentenceTransformer
from langchain_community.document_loaders import PyPDFDirectoryLoader
import numpy as np
import pymupdf
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter
import sqlite3
from langchain_text_splitters import RecursiveCharacterTextSplitter
import torch
import time

In [ ]:
t1 = time.time()
folder_path = Path.home()

# SQLite connection
conn = sqlite3.connect("Semantic_index.db")

c = conn.cursor()

# Create table
c.execute("""
CREATE TABLE IF NOT EXISTS File_info (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    File_name TEXT,
    File_location TEXT,
    Page_number INTEGER,
    Chunk_number INTEGER,
    FAISS_idx INTEGER
)
""")

# Text splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

texts = []

faiss_counter = 0

for pdf_file in folder_path.rglob("*.pdf"):

    # Skip empty files
    if pdf_file.stat().st_size == 0:
        print(f"Skipping empty file: {pdf_file.name}")
        continue

    try:

        doc = pymupdf.open(pdf_file)

        # Skip empty PDFs
        if doc.page_count == 0:
            print(f"Skipping PDF with 0 pages: {pdf_file.name}")
            doc.close()
            continue

        print(f"Processing: {pdf_file.name}")

        file_name = pdf_file.name

        file_location = str(pdf_file.resolve())

        folder_name = pdf_file.parent.name

        for page_num in range(doc.page_count):

            page = doc.load_page(page_num)

            text = page.get_text().strip()

            if not text:
                continue

            # --------------------------------
            # CHUNK PAGE TEXT
            # --------------------------------
            chunks = splitter.split_text(text)

            for chunk_num, chunk in enumerate(chunks):

                # --------------------------------
                # METADATA ENRICHMENT
                # --------------------------------
                rich_text = f"""
Filename: {file_name}

Folder: {folder_name}

File Path: {file_location}

Page Number: {page_num}

Chunk Number: {chunk_num}

Content:
{chunk}
"""

                texts.append(rich_text)

                # Insert metadata
                c.execute("""
                INSERT INTO File_info (
                    File_name,
                    File_location,
                    Page_number,
                    Chunk_number,
                    FAISS_idx
                )
                VALUES (?, ?, ?, ?, ?)
                """, (
                    file_name,
                    file_location,
                    page_num,
                    chunk_num,
                    faiss_counter
                ))

                faiss_counter += 1

        doc.close()

    except Exception as e:

        print(f"Failed: {pdf_file.name} -> {e}")

# Save DB
conn.commit()

conn.close()

print("Done indexing PDFs into SQLite.")

t2 = time.time()

print(t2 - t1)

## FAISS

### Writing the Index

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(
    "all-MiniLM-L6-v2",
    device=device
)
print(model.device)

In [ ]:
embeddings = model.encode(
    texts,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)
print(embeddings.shape)
vectors = np.array(embeddings).astype("float32")

dimension = vectors.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(vectors)

faiss.write_index(index, "file.index")

### Reading from the Index

In [ ]:
index = faiss.read_index("file.index")

print("Total vectors:", index.ntotal)

# -----------------------------
# SEARCH QUERY
# -----------------------------
query = "lab report"

query_vector = model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

# -----------------------------
# SEARCH FAISS
# -----------------------------
D, I = index.search(query_vector, k=200)

# -----------------------------
# SQLITE LOOKUP
# -----------------------------
conn = sqlite3.connect("Semantic_index.db")

c = conn.cursor()

seen_files = set()

unique_results = []

for i in range(len(I[0])):

    faiss_idx = int(I[0][i])

    similarity_score = float(D[0][i])

    c.execute("""
    SELECT *
    FROM File_info
    WHERE FAISS_idx = ?
    """, (faiss_idx,))

    result = c.fetchone()

    if result is None:
        continue

    file_name = result[1]

    file_location = result[2]

    page_number = result[3]

    # Skip duplicate files
    if file_location in seen_files:
        continue

    seen_files.add(file_location)

    unique_results.append({
        "file_name": file_name,
        "file_location": file_location,
        "page_number": page_number,
        "score": similarity_score
    })

    # Stop at 25 unique files
    if len(unique_results) == 25:
        break

conn.close()

# -----------------------------
# PRINT RESULTS
# -----------------------------
for idx, result in enumerate(unique_results):
    print(f"Rank #{idx + 1}")

    print("Similarity:", round(result["score"], 4))

    print("File Name:", result["file_name"])

    print("Location:", result["file_location"])

    print("Page:", result["page_number"])